# Avaliação: Assistente de IA Generativa com Gemini

> **Adaptação:** esta versão utiliza somente o Google Gemini porque o acesso ao Hugging Face apresentou erro no ambiente utilizado.

**Disciplina:** Disruptive Architectures: IoT, IoB & Generative AI

**Grupo:**
- Bruno Andrade Zanateli (RM563736)
- Christian Souza Freitas (RM569098)
- Rodrigo Tiezzi (RM562975)

**Tema escolhido:** Consultor de agricultura inteligente

**Escopo:** irrigação, umidade do solo e estações meteorológicas conectadas.

---

Neste notebook o grupo vai construir um assistente de IA em 4 etapas (e 1 bônus), conforme o README:

1. Assistente com **guardrails** via `system prompt`
2. Comparação de **temperatura 0.0 vs 1.0** com Hugging Face e Gemini
3. Chat com **memória e troca de modelo**, mantendo histórico compartilhado
4. Interface web com **Gradio**, seletor de modelo e slider de temperatura
5. (Bônus) API REST com **sessões independentes**

Os trechos marcados com **`# >>> PERSONALIZE`** foram adaptados ao tema do grupo.
Execute as células em ordem, de cima para baixo.

> **Segurança:** os tokens não ficam no código. Eles devem ser cadastrados nos Secrets do Google Colab como `HF_TOKEN` e `GEMINI_API_KEY`.

## 0. Configuração

In [1]:
# Instalação das dependências
# O notebook foi adaptado para usar somente o Google Gemini.
!pip install google-genai gradio fastapi uvicorn requests -q


In [3]:
from google import genai
from google.genai import types
from google.colab import userdata

# A chave fica nos Secrets do Colab (ícone de chave na barra lateral).
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

print("GEMINI_API_KEY ok" if GEMINI_API_KEY else "ERRO: adicione GEMINI_API_KEY nos Secrets do Colab")


GEMINI_API_KEY ok


In [4]:
# Modelo utilizado na versão Gemini Only
MODELO_GEMINI = "gemini-3.5-flash-lite"
cliente_gemini = genai.Client(api_key=GEMINI_API_KEY)


In [5]:
# >>> PERSONALIZE: tema do grupo
SYSTEM_PROMPT = """Você é um consultor de agricultura inteligente.
Ajude o usuário com dúvidas sobre irrigação, umidade do solo e estações meteorológicas conectadas.
Responda sempre em português, de forma clara e objetiva.
Não invente dados de sensores ou medições que não foram fornecidos.
Se a pergunta não tiver relação com agricultura inteligente, diga educadamente que não pode ajudar."""

---
## Etapa 1: Assistente com guardrails

Nesta etapa serão testados três `system prompts`:

1. **Sem restrição:** assistente geral.
2. **Com restrição de tema:** responde somente sobre agricultura inteligente.
3. **Com restrição de tema e formato:** responde somente sobre agricultura inteligente e em até 3 frases.

A mesma pergunta será enviada aos três sistemas. Também será feita uma pergunta fora do escopo para verificar o guardrail de tema.

In [6]:
def perguntar_gemini(pergunta, system, temperatura=0.7):
    """Envia uma pergunta ao Gemini com temperatura configurável."""
    resposta = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=pergunta,
        config=types.GenerateContentConfig(
            system_instruction=system,
            max_output_tokens=300,
            temperature=temperatura,
        ),
    )
    return resposta.text


In [7]:
# >>> PERSONALIZE: três guardrails para o tema do grupo
systems_guardrails = {
    "Sem restrição": (
        "Você é um assistente prestativo. Responda em português de forma clara."
    ),
    "Restrição de tema": (
        "Você é um consultor de agricultura inteligente. "
        "Responda somente perguntas sobre irrigação, umidade do solo e "
        "estações meteorológicas conectadas. Para perguntas fora desse tema, "
        "diga educadamente que não pode ajudar."
    ),
    "Tema + formato": (
        "Você é um consultor de agricultura inteligente. "
        "Responda somente perguntas sobre irrigação, umidade do solo e "
        "estações meteorológicas conectadas. Para perguntas fora desse tema, "
        "recuse educadamente. Responda sempre em no máximo 3 frases e em português."
    ),
}

pergunta_dentro = "Como um sensor de umidade do solo pode ajudar a controlar a irrigação?"
pergunta_fora = "Qual é a capital da França?"

for nome, system in systems_guardrails.items():
    print(f"\n===== {nome} | PERGUNTA DENTRO DO ESCOPO =====")
    print(perguntar_gemini(pergunta_dentro, system, temperatura=0.0))

    print(f"\n===== {nome} | PERGUNTA FORA DO ESCOPO =====")
    print(perguntar_gemini(pergunta_fora, system, temperatura=0.0))



===== Sem restrição | PERGUNTA DENTRO DO ESCOPO =====
Um sensor de umidade do solo é uma ferramenta fundamental para a **agricultura de precisão** e para a **jardinagem inteligente**. Ele funciona como um "termômetro" da terra, medindo a quantidade de água disponível na zona das raízes das plantas. 

Aqui está como ele ajuda a controlar a irrigação de forma eficiente:

### 1. Fim do desperdício de água
O maior problema da irrigação tradicional (baseada em horários fixos ou cronômetros) é que ela molha o solo mesmo quando não é necessário — como após uma chuva ou em dias nublados. O sensor de umidade evita isso, permitindo que o sistema de irrigação só seja ligado quando o solo realmente estiver seco. Isso pode gerar uma **economia de água de até 50%**.

### 2. Irrigação na medida certa (Evita excesso e falta)
* **Evita o estresse hídrico:** A planta não passa sede, pois o sensor avisa quando a umidade cai abaixo do limite ideal para aquela cultura específica.
* **Evita o encharcamento

**Observações do grupo (Etapa 1):**

- O guardrail de tema recusou a pergunta fora do escopo?
- A restrição de formato alterou o estilo/tamanho da resposta?
- Qual configuração foi mais adequada para o assistente?

**Resposta do grupo:**

1. No sistema sem restrição, o modelo pode responder perguntas gerais. Nos sistemas com restrição de tema, a pergunta sobre agricultura foi respondida, enquanto a pergunta fora do escopo foi recusada.
2. Sim. O terceiro sistema deixou as respostas mais curtas e objetivas, respeitando o limite de 3 frases.
3. A configuração **Tema + formato** foi a mais adequada porque mantém o assistente focado em agricultura inteligente e padroniza o tamanho das respostas.

---
## Etapa 2: Comparação com temperatura

Nesta versão, o teste de temperatura é realizado somente com o Gemini, pois o acesso ao Hugging Face foi removido.

Cada pergunta é enviada com `temperature=0.0` e `temperature=1.0`. A ideia é observar consistência com temperatura baixa e variedade com temperatura alta.


In [8]:
def perguntar_gemini(pergunta, system, temperatura=0.7):
    """Envia uma pergunta ao Gemini com temperatura configurável."""
    resposta = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=pergunta,
        config=types.GenerateContentConfig(
            system_instruction=system,
            max_output_tokens=300,
            temperature=temperatura,
        ),
    )
    return resposta.text

In [9]:
# >>> PERSONALIZE: 2 perguntas sobre o tema do grupo
perguntas = [
    "Como um sensor de umidade do solo pode otimizar o uso da água na irrigação?",
    "Como uma estação meteorológica conectada pode ajudar o produtor a decidir quando irrigar?",
]

for p in perguntas:
    print("\n" + "=" * 80)
    print("PERGUNTA:", p)

    for temperatura in [0.0, 1.0]:
        print(f"\n--- Gemini | temperature={temperatura} ---")
        print(perguntar_gemini(p, SYSTEM_PROMPT, temperatura=temperatura))



PERGUNTA: Como um sensor de umidade do solo pode otimizar o uso da água na irrigação?

--- Gemini | temperature=0.0 ---
Olá! Como consultor de agricultura inteligente, terei o maior prazer em explicar como essa tecnologia funciona.

Um sensor de umidade do solo otimiza o uso da água na irrigação das seguintes maneiras:

1. **Irrigação sob demanda:** Em vez de seguir um cronograma fixo (como regar todos os dias em horários predeterminados), o sistema utiliza os dados reais do sensor para irrigar apenas quando a umidade do solo atinge o limite crítico para a cultura.
2. **Prevenção do desperdício:** Evita a irrigação excessiva, garantindo que a água não seja aplicada quando o solo já está com níveis adequados de umidade, o que também previne a lixiviação de nutrientes.
3. **Economia de energia e recursos:** Ao bombear apenas a quantidade de água estritamente necessária, há uma redução direta no consumo de energia elétrica ou combustível utilizado pelos sistemas de bombeamento.
4. **Inte

**Observações do grupo (Etapa 2):**

- Com `temperature=0.0`, as respostas ficaram mais consistentes?
- Com `temperature=1.0`, ficaram mais variadas?
- Qual combinação de modelo e temperatura é mais adequada para agricultura inteligente?

**Resposta do grupo:**

1. A temperatura 0.0 tende a produzir respostas mais previsíveis e consistentes.
2. A temperatura 1.0 tende a permitir maior variedade na formulação das respostas.
3. Para agricultura inteligente, a combinação de temperatura baixa com um modelo que apresente explicações claras é mais adequada, principalmente quando a resposta precisa ser objetiva e confiável.

---
## Etapa 3: Chat com memória

Como o Hugging Face foi removido nesta versão, o chat utiliza somente o Gemini. O histórico continua sendo armazenado pelo código para demonstrar memória de conversa.

Comandos:
- `sair` encerra o chat
- `limpar` apaga o histórico e mantém somente o `system`
- `historico` mostra as mensagens armazenadas

**Teste sugerido:**
1. Diga seu nome.
2. Pergunte qual é o seu nome.
3. Faça outra pergunta que dependa da conversa anterior.
4. Use `limpar` e pergunte novamente.


In [10]:
historico = [{"role": "system", "content": SYSTEM_PROMPT}]

def responder_com_historico():
    global historico

    conteudos = []
    for msg in historico:
        if msg["role"] == "user":
            conteudos.append({"role": "user", "parts": [{"text": msg["content"]}]})
        elif msg["role"] == "assistant":
            conteudos.append({"role": "model", "parts": [{"text": msg["content"]}]})

    resposta = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=conteudos,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            max_output_tokens=300,
            temperature=0.0,
        ),
    )
    return resposta.text

print("Chat iniciado!")
print("Comandos: sair | limpar | historico")
print("Modelo atual: Gemini")

while True:
    entrada = input("Você: " ).strip()

    if entrada.lower() == "sair":
        print("Encerrando chat.")
        break

    if entrada.lower() == "limpar":
        historico = [{"role": "system", "content": SYSTEM_PROMPT}]
        print("\n(histórico apagado)\n")
        continue

    if entrada.lower() == "historico":
        print(f"\n(mensagens no histórico: {len(historico)})\n")
        continue

    historico.append({"role": "user", "content": entrada})
    texto = responder_com_historico()
    historico.append({"role": "assistant", "content": texto})

    print(f"\n[Gemini] Assistente: {texto}\n")


Chat iniciado!
Comandos: sair | limpar | historico
Modelo atual: Gemini
Você: my names is Bruno!

[Gemini] Assistente: Olá, Bruno! É um prazer falar com você. 

Como seu consultor de agricultura inteligente, estou aqui para ajudar com dúvidas sobre sistemas de irrigação, monitoramento da umidade do solo, uso de estações meteorológicas conectadas e outras tecnologias para o campo. 

Como posso te ajudar hoje?

Você: Do you remember my name?

[Gemini] Assistente: Sim, claro! Seu nome é Bruno. 

Como posso ajudar você e sua produção hoje com irrigação, umidade do solo ou estações meteorológicas?

Você: limpar

(histórico apagado)

Você: whats my name?

[Gemini] Assistente: Como sou um consultor de agricultura inteligente, não tenho acesso a informações pessoais, portanto não sei o seu nome. 

Como posso ajudar você hoje com irrigação, umidade do solo ou estações meteorológicas?

Você: sair
Encerrando chat.


In [11]:
# Veja como ficou a lista enviada ao modelo
for msg in historico:
    print(f"[{msg['role']}] {msg['content'][:80].replace(chr(10), ' ')}")

[system] Você é um consultor de agricultura inteligente. Ajude o usuário com dúvidas sobr
[user] whats my name?
[assistant] Como sou um consultor de agricultura inteligente, não tenho acesso a informações


**Observações do grupo (Etapa 3):**

- Após trocar de modelo, o novo modelo lembrou do que havia sido dito?
- Por que isso acontece?
- O que aconteceu depois de usar `limpar`?

**Resposta do grupo:**

1. Sim. Após a troca, o novo modelo recebeu o histórico compartilhado e pôde usar as informações anteriores da conversa.
2. Isso acontece porque a memória não pertence ao modelo: o código mantém a lista `historico` e envia as mensagens anteriores para o modelo selecionado.
3. Depois de `limpar`, somente o `system prompt` permanece no histórico. Portanto, informações dadas anteriormente deixam de ser enviadas ao modelo.

---
## Etapa 4: Interface web com Gradio

A interface utiliza somente o Gemini e permite controlar a temperatura com um slider de `0.0` a `1.0`.

**Após executar a célula abaixo no Colab, teste temperaturas baixa e alta e tire um print da interface com o slider visível para salvar em `prints/gradio.png`.


In [12]:
import gradio as gr

def texto_da_mensagem(conteudo):
    """Extrai o texto de uma mensagem do histórico do Gradio."""
    if isinstance(conteudo, str):
        return conteudo
    if isinstance(conteudo, list):
        textos = []
        for item in conteudo:
            if isinstance(item, dict):
                textos.append(item.get("text", ""))
        return " ".join(textos)
    return str(conteudo)

def responder(mensagem, historico_gradio, temperatura):
    conteudos = []
    for msg in historico_gradio:
        conteudos.append({
            "role": "model" if msg["role"] == "assistant" else "user",
            "parts": [{"text": texto_da_mensagem(msg["content"])}]
        })
    conteudos.append({"role": "user", "parts": [{"text": mensagem}]})

    resposta = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=conteudos,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            max_output_tokens=300,
            temperature=temperatura,
        ),
    )
    return resposta.text

interface = gr.ChatInterface(
    fn=responder,
    additional_inputs=[
        gr.Slider(
            minimum=0.0,
            maximum=1.0,
            value=0.0,
            step=0.1,
            label="Temperatura"
        ),
    ],
    title="Consultor de Agricultura Inteligente — Gemini",
    description="Pergunte sobre irrigação, umidade do solo e estações meteorológicas conectadas.",
    examples=[
        ["Como funciona um sistema com sensor de umidade e irrigação automática?", 0.0],
        ["Como uma estação meteorológica pode ajudar na irrigação?", 0.5],
        ["Quais dados um sensor de umidade do solo pode fornecer?", 1.0],
    ],
)

interface.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://fcf06738927b6247f7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**Observação:** a interface principal já foi criada na célula anterior. Não execute uma segunda interface durante `Executar tudo`; use a célula anterior para o teste e o print solicitado.


**Observações do grupo (Etapa 4):**

- Tire um print da interface funcionando com o slider de temperatura visível e salve como `prints/gradio.png`.
- Troque de modelo no meio da conversa. O assistente continuou lembrando do que foi dito? Por quê?
- Compare uma resposta com temperatura baixa e outra com temperatura alta.

**Resposta do grupo:**

1. Os dois modelos conseguem utilizar o histórico enviado pela interface, mas cada chamada continua sendo feita pelo modelo selecionado.
2. A memória da conversa é mantida pelo histórico que o Gradio entrega à função `responder`; ela não é uma memória permanente do modelo.
3. Com temperatura baixa, esperamos respostas mais previsíveis e consistentes. Com temperatura alta, esperamos maior variedade na forma de responder.

---
## Etapa 5 (Bônus): API com FastAPI e sessões independentes

Nesta versão, a API utiliza somente o Gemini e mantém um histórico separado para cada `session_id`.

Teste:
- **Sessão A:** diga o nome `Bruno` e depois pergunte qual é o nome.
- **Sessão B:** pergunte qual é o nome sem ter informado nada.
- A sessão A deve lembrar; a sessão B não deve ter acesso ao histórico da A.


In [13]:
%%writefile app_avaliacao.py
import os
from fastapi import FastAPI
from pydantic import BaseModel
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

MODELO_GEMINI = "gemini-3.5-flash-lite"
SYSTEM_PROMPT = os.environ.get(
    "SYSTEM_PROMPT",
    "Você é um consultor de agricultura inteligente. Responda em português."
)

app = FastAPI()
sessoes = {}

class Pergunta(BaseModel):
    session_id: str
    mensagem: str

@app.post("/chat")
def chat(pergunta: Pergunta):
    if pergunta.session_id not in sessoes:
        sessoes[pergunta.session_id] = []

    historico = sessoes[pergunta.session_id]
    historico.append({"role": "user", "content": pergunta.mensagem})

    conteudos = []
    for msg in historico:
        conteudos.append({
            "role": "user" if msg["role"] == "user" else "model",
            "parts": [{"text": msg["content"]}]
        })

    resposta = client.models.generate_content(
        model=MODELO_GEMINI,
        contents=conteudos,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            max_output_tokens=300,
            temperature=0.0,
        ),
    )
    texto = resposta.text
    historico.append({"role": "assistant", "content": texto})

    return {"session_id": pergunta.session_id, "resposta": texto}

@app.delete("/chat/{session_id}")
def limpar_sessao(session_id: str):
    sessoes.pop(session_id, None)
    return {"session_id": session_id, "status": "sessão removida"}


Writing app_avaliacao.py


In [14]:
# Inicia o servidor em segundo plano, dentro do próprio Colab
!pip install fastapi uvicorn requests google-genai -q

import os, subprocess, time

# Os valores vêm dos Secrets; nenhum token é gravado no notebook.
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ["SYSTEM_PROMPT"] = SYSTEM_PROMPT

servidor = subprocess.Popen(["uvicorn", "app_avaliacao:app", "--port", "8000"])
time.sleep(5)
print("Servidor rodando em http://localhost:8000")


Servidor rodando em http://localhost:8000


In [15]:
# Teste de duas sessões independentes
import requests

url = "http://localhost:8000/chat"

# Sessão A: primeiro informa o nome
r1 = requests.post(url, json={
    "session_id": "A",
    "mensagem": "Meu nome é Bruno."
})
print("Sessão A - primeira resposta:", r1.json()["resposta"])

# Sessão A: deve lembrar
r2 = requests.post(url, json={
    "session_id": "A",
    "mensagem": "Qual é o meu nome?"
})
print("Sessão A - memória:", r2.json()["resposta"])

# Sessão B: não recebeu a conversa da A
r3 = requests.post(url, json={
    "session_id": "B",
    "mensagem": "Qual é o meu nome?"
})
print("Sessão B - independente:", r3.json()["resposta"])

Sessão A - primeira resposta: Olá, Bruno! É um prazer falar com você. 

Como consultor de agricultura inteligente, estou aqui para ajudar com suas dúvidas sobre sistemas de irrigação, monitoramento da umidade do solo, estações meteorológicas conectadas e outras tecnologias para o campo. 

Como posso ajudar você hoje?
Sessão A - memória: Seu nome é Bruno. Como posso ajudar você hoje com seus projetos de agricultura inteligente?
Sessão B - independente: Como um consultor de agricultura inteligente, eu não tenho acesso a informações pessoais, portanto não sei o seu nome. 

Como posso ajudar você hoje com irrigação, umidade do solo ou estações meteorológicas?


In [16]:
# Encerra o servidor
servidor.terminate()
servidor.wait()
print("Servidor encerrado.")

Servidor encerrado.


### Observações do grupo (Etapa 5 - bônus)

- A sessão A respondeu o nome corretamente?
- A sessão B teve acesso ao nome informado na sessão A?
- O que diferencia esta API da atividade anterior?

**Resposta do grupo:**

1. A sessão A deve lembrar o nome porque o histórico é armazenado usando a chave `session_id="A"`.
2. A sessão B não deve saber o nome informado na sessão A porque possui um histórico separado, associado a `session_id="B"`.
3. A principal diferença é que a API agora mantém **sessões independentes**, permitindo que diferentes usuários/conversas tenham memórias separadas.
